# 04 Portfolio Risk and Scenarios

### Objective
Summarize Expected Loss concentration and demonstrate deterministic PD/LGD/EAD scenario analysis.

### Inputs
- `data/demo/loans_with_pd_lgd_ead_sample_v1.csv`
- `data/processed/expected_loss_top_concentration_v1.csv`
- `data/processed/expected_loss_pd_ead_matrix_v1.csv`

### Methodology
Portfolio concentration is evaluated by loan-level Expected Loss rank and by PD x EAD quartile cells. Scenario analysis applies deterministic multipliers to PD, LGD, and EAD.

### Key Outputs
- Expected Loss concentration table
- PD x EAD driver matrix
- Deterministic scenario summary

In [1]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

reports = project_root / 'reports'
tables = reports / 'tables'
processed = project_root / 'data' / 'processed'

from credit_risk.scenarios import deterministic_expected_loss_scenario

loans = pd.read_csv(project_root / 'data' / 'demo' / 'loans_with_pd_lgd_ead_sample_v1.csv')
concentration = pd.read_csv(processed / 'expected_loss_top_concentration_v1.csv')
matrix = pd.read_csv(processed / 'expected_loss_pd_ead_matrix_v1.csv')

In [2]:
concentration

,top_group,loan_count,share_of_loans,total_expected_loss,share_of_total_expected_loss,total_ead,share_of_total_ead,mean_predicted_pd,mean_ead,mean_expected_loss
0,top_1pct_loans_by_expected_loss,13454,0.01,2.063858e+08,0.053432,4.476286e+08,0.023087,0.502570,33271.038353,15340.108019
1,top_5pct_loans_by_expected_loss,67268,0.05,7.870600e+08,0.203763,2.026850e+09,0.104538,0.428156,30130.969034,11700.362696
2,top_10pct_loans_by_expected_loss,134535,0.10,1.319203e+09,0.341530,3.737454e+09,0.192766,0.392608,27780.529230,9805.648223
3,top_20pct_loans_by_expected_loss,269070,0.20,2.081277e+09,0.538825,6.634674e+09,0.342195,0.351713,24657.797878,7735.076339


In [3]:
matrix.sort_values('total_expected_loss', ascending=False).head(8)

,predicted_pd_quartile,ead_proxy_quartile,loan_count,mean_predicted_pd,mean_ead,total_ead,mean_expected_loss,total_expected_loss,share_of_total_expected_loss
15,Q4_highest_pd,Q4_highest_ead,113643,0.382972,28054.100341,3.188152e+09,9933.112726,1.128829e+09,0.292244
14,Q4_highest_pd,Q3,110540,0.374892,16480.749276,1.821782e+09,5713.579993,6.315791e+08,0.163510
11,Q3,Q4_highest_ead,67257,0.225124,28153.453172,1.893517e+09,5860.494267,3.941593e+08,0.102044
10,Q3,Q3,87156,0.223632,16453.263401,1.434001e+09,3402.097129,2.965132e+08,0.076765
13,Q4_highest_pd,Q2,61751,0.359568,10613.535003,6.553964e+08,3540.157880,2.186083e+08,0.056596
7,Q2,Q4_highest_ead,53088,0.142353,27754.776503,1.473446e+09,3656.136126,1.940970e+08,0.050250
6,Q2,Q3,83848,0.141083,16503.702533,1.383802e+09,2152.968333,1.805221e+08,0.046736
9,Q3,Q2,74204,0.221386,10370.215891,7.695115e+08,2123.381406,1.575634e+08,0.040792


In [4]:
scenario = deterministic_expected_loss_scenario(loans, pd_multiplier=1.25, lgd_multiplier=1.10, ead_multiplier=1.00)

pd.DataFrame({
    'metric': ['baseline_expected_loss', 'scenario_expected_loss', 'difference'],
    'value': [loans['expected_loss_prelim'].sum(), scenario['scenario_expected_loss'].sum(), scenario['scenario_expected_loss'].sum() - loans['expected_loss_prelim'].sum()],
})

,metric,value
0,baseline_expected_loss,2.862190e+08
1,scenario_expected_loss,3.868865e+08
2,difference,1.006675e+08


### Findings
- Expected Loss is concentrated in high-PD and high-EAD segments.
- Deterministic scenario analysis shows how PD, LGD, and EAD shocks mechanically transmit into Expected Loss.

### Limitations
- The scenario layer is deterministic and does not estimate macroeconomic relationships.
- The demo sample supports dashboard deployment and does not replace full local portfolio analysis.